In [16]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import json

In [17]:
df = pd.read_csv('assets/dataset.csv')

# print columns so you can check exact names
print(df.columns.tolist())
print(df.head())

['age', 'gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'diabetes_risk_score', 'diabetes_stage', 'diagnosed_diabetes']
   age  gender ethnicity education_level  income_level employment_status  \
0   58    Male     Asian      Highschool  Lower-Middle          Employed   
1   48  Female     White      Highschool        Middle          Employed   
2   60    Male  Hispanic      Highschool        Middle        Unemployed   
3   74  Female     Black      Highschool           Low           Retired   
4   46    Male     W

In [18]:
smoking_dummies

,smoking_Current,smoking_Former,smoking_Never
0,False,False,True
1,False,True,False
2,False,False,True
3,False,False,True
4,False,False,True
...,...,...,...
99995,False,True,False
99996,False,False,True
99997,False,True,False
99998,False,False,True


In [19]:
# ── one-hot encode smoking ────────────────────────

smoking_dummies = pd.get_dummies(
    df['smoking_status'],
    prefix='smoking',
    drop_first=False
)

# never is the baseline
df['smoking_former']  = smoking_dummies.get('smoking_Former',  0).astype(int)
df['smoking_current'] = smoking_dummies.get('smoking_Current', 0).astype(int)

# ── define features ───────────────────────────────
# smoking_former and smoking_current replace smoking_status
FEATURES = [
    'smoking_former',
    'smoking_current',
    'alcohol_consumption_per_week',
    'physical_activity_minutes_per_week',
    'diet_score',
    'sleep_hours_per_day',
    'screen_time_hours_per_day',
]
TARGET = 'diagnosed_diabetes'

df = df.dropna(subset=FEATURES + [TARGET])

X = df[FEATURES].values
y = df[TARGET].values

In [20]:
# ── train ─────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

print('Accuracy:', model.score(X_scaled, y))
print('Coefficients:', dict(zip(FEATURES, model.coef_[0])))

Accuracy: 0.6039
Coefficients: {'smoking_former': 0.0012138526557888341, 'smoking_current': 0.0017833685293771166, 'alcohol_consumption_per_week': 0.0010747585923089335, 'physical_activity_minutes_per_week': -0.2046540802097735, 'diet_score': -0.0921445269683404, 'sleep_hours_per_day': -0.001812152943164593, 'screen_time_hours_per_day': 0.03783000009230646}


In [21]:
# ── export ────────────────────────────────────────
export = {
    'features':     FEATURES,
    'coefficients': model.coef_[0].tolist(),
    'intercept':    float(model.intercept_[0]),
    'means':        scaler.mean_.tolist(),
    'stds':         scaler.scale_.tolist(),
}

with open('assets/lifestyle_model.js', 'w') as f:
    f.write('const MODEL = ')
    f.write(json.dumps(export, indent=2))
    f.write(';\n')

print('model.js written')

model.js written
